# Apache Hive — MovieLens SQL Analytics Lab

> Standardized portfolio notebook. Original lab document and manually uploaded evidence notebook remain preserved.

**Domain:** Data Warehousing / SQL on Hadoop  
**Exercise type:** Hive SQL analytics over MovieLens data  
**Tags:** `apache-hive`, `hadoop`, `sql`, `movielens`, `beeline`, `derby-metastore`, `analytics`


## Narrative

This lab demonstrates Hive as the SQL abstraction over Hadoop. The workflow restarts Hadoop services, initializes Hive metadata, starts Beeline, loads MovieLens data, transforms Unix timestamps into weekdays, and queries rating activity.

Use cases include SQL-on-HDFS analytics, data warehousing fundamentals, batch feature preparation, and migration planning to Spark SQL, Trino, Athena, Databricks SQL, or lakehouse table formats.


## Step 1 — Restart Hadoop Services

The lab stops and restarts HDFS/YARN before Hive work begins. This validates that distributed storage and resource management are available.


In [ ]:
stop-dfs.sh
stop-yarn.sh
start-dfs.sh
start-yarn.sh


## Step 2 — Initialize Hive Metastore and Beeline

The original runbook uses `schemaTool` to initialize the metastore, then connects through Beeline on localhost. This represents the metadata layer Hive needs before tables can be created and queried.


In [ ]:
schematool -dbType derby -initSchema
beeline -u jdbc:hive2://localhost:10000
show tables;


## Step 3 — Load MovieLens Data

The lab creates a table such as `u_data`, loads rating records, and validates row retrieval. This is a classic staging-table pattern.


In [ ]:
CREATE TABLE u_data (userid INT, movieid INT, rating INT, unixtime STRING)
ROW FORMAT DELIMITED FIELDS TERMINATED BY '\t';
LOAD DATA LOCAL INPATH '<path-to-u.data>' OVERWRITE INTO TABLE u_data;
SELECT * FROM u_data LIMIT 10;


## Step 4 — Transform Timestamp to Weekday

The lab applies a Python mapper to convert Unix time into weekday labels and then queries records by weekday. The domain idea is feature enrichment: adding an interpretable time dimension to raw event data.


In [ ]:
ADD FILE weekday_mapper.py;
CREATE TABLE u_data_new AS
SELECT TRANSFORM (userid, movieid, rating, unixtime)
USING 'python weekday_mapper.py'
AS (userid, movieid, rating, weekday)
FROM u_data;
SELECT weekday, COUNT(*) FROM u_data_new GROUP BY weekday;


## Validation

- Confirm Hadoop services start cleanly.
- Confirm Hive metastore initializes.
- Confirm Beeline connects.
- Confirm tables are created and populated.
- Confirm weekday transformation produces grouped outputs.


## Modernization Notes

Hive teaches SQL-on-Hadoop fundamentals. Modern equivalents include Spark SQL, Trino/Presto, Athena, BigQuery, Snowflake, Databricks SQL, and lakehouse tables such as Iceberg, Delta Lake, or Hudi. The durable concept is table metadata plus SQL transformation over distributed data.
